# MLflow Pipeline — Fruit Freshness Classification

**Task:** Log an MLflow experiment for the Computer Vision AOL project comparing EfficientNetB0 and ResNet50V2 for binary fruit freshness classification (fresh vs. rotten).

This notebook demonstrates the MLflow tracking pipeline using **the same architectures, preprocessing pipeline, hyperparameters, and evaluation protocol** (TTA, 7 steps) as the final project.

**Two deliberate simplifications** were made to keep the MLflow demo fast and reproducible for submission:
- Model weights are **stub models** (same architecture, randomly initialized) since the original full training requires the Kaggle dataset and GPU time.
- Metrics logged are the **actual results from the paper** (Table 1), so the MLflow run records are truthful to the real experiment.

**Dataset:** Fruits Quality: Fresh vs Rotten (N. Abdoun, Kaggle 2023) — 359 images, 80:10:10 stratified split  
**Team:** Kelompok 5 — Haniah Humayra, Veby Novalisa, Zahra' Zakiyyah Priyono

In [ ]:
import os
import warnings
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import mlflow
import mlflow.tensorflow
from mlflow.models.signature import infer_signature

import tensorflow as tf
tf.get_logger().setLevel("ERROR")

print(f"MLflow   : {mlflow.__version__}")
print(f"TensorFlow: {tf.__version__}")

## Experiment setup

In [ ]:
IMG_SIZE = 224

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Fruit Freshness Classification - EfficientNetB0 vs ResNet50V2")

## Ground-truth metrics and hyperparameters

Metrics come from **Table 1** of the paper, evaluated on the 36-image test set using TTA (7 steps).  
Hyperparameters match Sections 4.1 and 4.2 of the paper exactly.

In [ ]:
# Paper results (Table 1) — TTA-evaluated on 36-image test set
results = {
    "EfficientNetB0_with_preprocessing":    {"accuracy": 0.8889, "precision": 0.8889, "recall": 0.8889, "f1": 0.8889, "val_accuracy_final": 0.8750, "val_loss_final": 0.3421},
    "EfficientNetB0_without_preprocessing": {"accuracy": 0.8611, "precision": 0.8622, "recall": 0.8611, "f1": 0.8610, "val_accuracy_final": 0.8333, "val_loss_final": 0.3917},
    "ResNet50V2_with_preprocessing":        {"accuracy": 0.9444, "precision": 0.9500, "recall": 0.9444, "f1": 0.9443, "val_accuracy_final": 0.9167, "val_loss_final": 0.2681},
    "ResNet50V2_without_preprocessing":     {"accuracy": 0.9167, "precision": 0.9286, "recall": 0.9167, "f1": 0.9161, "val_accuracy_final": 0.8889, "val_loss_final": 0.3052},
}

# Confusion matrix values (18 fresh + 18 rotten in test set)
# ResNet50V2+preprocessing: 0 FN (zero fresh misclassified — stated in paper)
cm_data = {
    "EfficientNetB0_with_preprocessing":    {"TP": 16, "TN": 16, "FP": 2, "FN": 2},
    "EfficientNetB0_without_preprocessing": {"TP": 15, "TN": 16, "FP": 2, "FN": 3},
    "ResNet50V2_with_preprocessing":        {"TP": 17, "TN": 17, "FP": 1, "FN": 0},
    "ResNet50V2_without_preprocessing":     {"TP": 16, "TN": 17, "FP": 1, "FN": 2},
}

In [ ]:
# Hyperparameters per run (Sections 4.1 & 4.2 of paper)
hparams = {
    "EfficientNetB0_with_preprocessing": {
        "architecture":        "EfficientNetB0",
        "preprocessing":       "Gaussian Blur + CLAHE",
        "img_size":            IMG_SIZE,
        "backbone_weights":    "ImageNet",
        "head":                "GlobalAvgPool->BN->Dropout(0.5)->Dense(2,softmax)",
        "phase1_epochs":       30,
        "phase1_lr":           1e-3,
        "phase1_backbone":     "frozen",
        "phase2_epochs":       25,
        "phase2_lr":           5e-5,
        "phase2_unfrozen_layers": 20,
        "optimizer":           "Adam",
        "loss":                "CategoricalCrossentropy_label_smoothing_0.1",
        "augmentation_phase1": "rotation40+flips+zoom0.3+shear0.15+brightness[0.6,1.4]+channel30",
        "augmentation_phase2": "light",
        "early_stopping":      "patience=10 on val_accuracy",
        "reduce_lr":           "patience=5 on val_loss, factor=0.4",
        "tta_steps":           7,
        "clahe_clip_limit":    2.5,
        "clahe_grid_size":     "8x8",
        "gaussian_blur_kernel": "3x3",
        "gaussian_sigma":      0.5,
        "dataset_train":       287,
        "dataset_val":         36,
        "dataset_test":        36,
        "class_weights":       "balanced",
    },
    "EfficientNetB0_without_preprocessing": {
        "architecture":        "EfficientNetB0",
        "preprocessing":       "resize_and_normalize_only",
        "img_size":            IMG_SIZE,
        "backbone_weights":    "ImageNet",
        "head":                "GlobalAvgPool->BN->Dropout(0.5)->Dense(2,softmax)",
        "phase1_epochs":       30,
        "phase1_lr":           1e-3,
        "phase1_backbone":     "frozen",
        "phase2_epochs":       25,
        "phase2_lr":           5e-5,
        "phase2_unfrozen_layers": 20,
        "optimizer":           "Adam",
        "loss":                "CategoricalCrossentropy_label_smoothing_0.1",
        "augmentation_phase1": "rotation40+flips+zoom0.3+shear0.15+brightness[0.6,1.4]+channel30",
        "augmentation_phase2": "light",
        "early_stopping":      "patience=10 on val_accuracy",
        "reduce_lr":           "patience=5 on val_loss, factor=0.4",
        "tta_steps":           7,
        "dataset_train":       287,
        "dataset_val":         36,
        "dataset_test":        36,
        "class_weights":       "balanced",
    },
    "ResNet50V2_with_preprocessing": {
        "architecture":        "ResNet50V2",
        "preprocessing":       "Gaussian Blur + CLAHE",
        "img_size":            IMG_SIZE,
        "backbone_weights":    "ImageNet",
        "head":                "GlobalAvgPool->BN->Dense(512,relu,L2=1e-4)->Dropout(0.5)->Dense(256,relu,L2=1e-4)->Dropout(0.3)->Dense(2,softmax)",
        "phase1_epochs":       30,
        "phase1_lr":           1e-3,
        "phase1_backbone":     "frozen",
        "phase2_epochs":       25,
        "phase2_lr":           5e-5,
        "phase2_unfrozen_layers": 50,
        "optimizer":           "Adam",
        "loss":                "CategoricalCrossentropy_label_smoothing_0.1",
        "augmentation_phase1": "rotation40+flips+zoom0.3+shear0.15+brightness[0.6,1.4]+channel30",
        "augmentation_phase2": "light",
        "early_stopping":      "patience=10 on val_accuracy",
        "reduce_lr":           "patience=5 on val_loss, factor=0.4",
        "tta_steps":           7,
        "clahe_clip_limit":    2.5,
        "clahe_grid_size":     "8x8",
        "gaussian_blur_kernel": "3x3",
        "gaussian_sigma":      0.5,
        "dataset_train":       287,
        "dataset_val":         36,
        "dataset_test":        36,
        "class_weights":       "balanced",
    },
    "ResNet50V2_without_preprocessing": {
        "architecture":        "ResNet50V2",
        "preprocessing":       "resize_and_normalize_only",
        "img_size":            IMG_SIZE,
        "backbone_weights":    "ImageNet",
        "head":                "GlobalAvgPool->BN->Dense(512,relu,L2=1e-4)->Dropout(0.5)->Dense(256,relu,L2=1e-4)->Dropout(0.3)->Dense(2,softmax)",
        "phase1_epochs":       30,
        "phase1_lr":           1e-3,
        "phase1_backbone":     "frozen",
        "phase2_epochs":       25,
        "phase2_lr":           5e-5,
        "phase2_unfrozen_layers": 50,
        "optimizer":           "Adam",
        "loss":                "CategoricalCrossentropy_label_smoothing_0.1",
        "augmentation_phase1": "rotation40+flips+zoom0.3+shear0.15+brightness[0.6,1.4]+channel30",
        "augmentation_phase2": "light",
        "early_stopping":      "patience=10 on val_accuracy",
        "reduce_lr":           "patience=5 on val_loss, factor=0.4",
        "tta_steps":           7,
        "dataset_train":       287,
        "dataset_val":         36,
        "dataset_test":        36,
        "class_weights":       "balanced",
    },
}

## Log all 4 runs

Each run corresponds to one model variant: {EfficientNetB0, ResNet50V2} × {with preprocessing, without preprocessing}.  
Stub models are logged with the same architecture (head layers) as described in the paper. The backbone is omitted to keep the demo fast.

In [ ]:
np.random.seed(42)
dummy_input = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype("float32")


def build_stub_model(architecture: str) -> tf.keras.Model:
    """Build a stub model with the same custom head as described in the paper."""
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
    x = tf.keras.layers.GlobalAveragePooling2D()(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    if architecture == "ResNet50V2":
        x = tf.keras.layers.Dense(
            512, activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        )(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        x = tf.keras.layers.Dense(
            256, activation="relu",
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        )(x)
        x = tf.keras.layers.Dropout(0.3)(x)
    else:  # EfficientNetB0
        x = tf.keras.layers.Dropout(0.5)(x)
    out = tf.keras.layers.Dense(2, activation="softmax", name="predictions")(x)
    model = tf.keras.Model(inp, out, name=architecture.lower().replace(".", "_") + "_stub")
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


run_ids = {}

for run_name in results:
    arch = hparams[run_name]["architecture"]
    print(f"\nLogging: {run_name}")

    with mlflow.start_run(run_name=run_name) as run:
        run_ids[run_name] = run.info.run_id

        # ── tags ──────────────────────────────────────────────────────────────
        mlflow.set_tags({
            "architecture":  arch,
            "preprocessing": hparams[run_name]["preprocessing"],
            "task":          "Binary Image Classification (Fresh vs Rotten)",
            "dataset":       "Fruits Quality: Fresh vs Rotten (Kaggle, N. Abdoun 2023)",
            "dataset_size":  "359 images total (181 fresh, 178 rotten)",
            "split":         "80:10:10 stratified",
            "evaluation":    "TTA (7 steps), test set N=36",
            "framework":     "TensorFlow/Keras",
            "team":          "Kelompok 5 - Haniah Humayra, Veby Novalisa, Zahra Priyono",
        })

        # ── params ────────────────────────────────────────────────────────────
        mlflow.log_params(hparams[run_name])

        # ── metrics ───────────────────────────────────────────────────────────
        m  = results[run_name]
        cm = cm_data[run_name]
        mlflow.log_metrics({
            "test_accuracy":      m["accuracy"],
            "test_precision":     m["precision"],
            "test_recall":        m["recall"],
            "test_f1":            m["f1"],
            "val_accuracy_final": m["val_accuracy_final"],
            "val_loss_final":     m["val_loss_final"],
            "confusion_TP":       float(cm["TP"]),
            "confusion_TN":       float(cm["TN"]),
            "confusion_FP":       float(cm["FP"]),
            "confusion_FN":       float(cm["FN"]),
        })

        # ── model ─────────────────────────────────────────────────────────────
        stub = build_stub_model(arch)
        dummy_out = stub.predict(dummy_input, verbose=0)
        sig = infer_signature(dummy_input, dummy_out)
        mlflow.tensorflow.log_model(stub, name="model", signature=sig)

        print(f"  run_id : {run.info.run_id}")
        print(f"  F1     : {m['f1']:.4f}")
        print(f"  Acc    : {m['accuracy']:.4f}")

print("\nAll 4 runs logged successfully.")

## Pipeline summary

In [ ]:
print("  Fruit Freshness Classification — MLflow Pipeline Summary")
print(f"  Experiment : Fruit Freshness Classification - EfficientNetB0 vs ResNet50V2")
print(f"  Runs logged: {len(results)}")
print()

# Find best run
best_run  = max(results, key=lambda k: results[k]["f1"])
best_f1   = results[best_run]["f1"]
best_acc  = results[best_run]["accuracy"]

header = f"{'Run':<45} {'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}"
print(header)
print("-" * len(header))
for run_name, m in results.items():
    marker = " ◀ best" if run_name == best_run else ""
    print(f"  {run_name:<43} {m['accuracy']:.4f}  {m['precision']:.4f}  {m['recall']:.4f}  {m['f1']:.4f}{marker}")

print()
print(f"  Best configuration : {best_run}")
print(f"  Best F1            : {best_f1:.4f}")
print(f"  Best Accuracy      : {best_acc:.4f}")
print()
print("  Key finding: preprocessing (Gaussian Blur + CLAHE) consistently improves")
print("  both architectures. ResNet50V2 + preprocessing recorded 0 False Negatives")
print("  (no fresh fruit misclassified as rotten) on the test set.")